In [1]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install gymnasium sb3-contrib stable-baselines3 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 6.6 MB/s eta 0:00:00


In [2]:
# ── 2. System path setup ─────────────────────────────────────────────────────
import sys, os
import numpy as np

DATASET_PATH = "/kaggle/input/datasets/nguyennhatphong/roguelike-rl-env"
sys.path.insert(0, DATASET_PATH)

print("Python version:", sys.version)
print("Dataset files:", os.listdir(DATASET_PATH)[:10])

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Dataset files: ['python_roguelike']


In [ ]:
# ── 3. Defensive Environment Wrapper ─────────────────────────────────────────
from python_roguelike.env.roguelike_env import RoguelikeEnv
from python_roguelike.data.enums import GameState
from sb3_contrib.common.wrappers import ActionMasker


class DefensiveEnv(RoguelikeEnv):
    """
    Defensive agent reward shaping (v4):
    - HP-loss penalty reduced from 0.5 → 0.15 (was overwhelming floor signal)
    - Floor reward raised from +10 → +25 (now competes with HP penalty)
    - Enemy kill reward +5 per kill (direct combat engagement signal)
    - Removed ★1 card penalty (harmful early-game)
    - ent_coef raised to 0.10 (prevents premature entropy collapse)
    - Win = +500, Loss = −200 (unified)
    """

    def __init__(self, seed=42, json_path=None, max_steps=10_000):
        super().__init__(seed=seed, json_path=json_path, max_steps=max_steps)
        self._prev_deck_size = 0
        self._prev_enemy_count = 0

    def reset(self, *, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self._prev_deck_size = len(self._controller.current_run.the_hero.deck.master_deck)
        self._prev_enemy_count = 0
        return obs, info

    def _count_alive_enemies(self):
        combat = self._controller.current_run.current_combat
        if combat is None:
            return 0
        return sum(1 for e in combat.enemies if e.current_health > 0)

    def step(self, action: int):
        run = self._controller.current_run
        prev_hp = run.the_hero.current_health
        prev_gold = run.the_hero.current_gold
        prev_floor = run.current_floor
        prev_in_combat = (run.current_state == GameState.InCombat)
        prev_alive = self._count_alive_enemies()

        obs, _base_reward, terminated, truncated, info = super().step(action)

        run = self._controller.current_run
        hero = run.the_hero
        reward = 0.0

        # HP change — asymmetric but rebalanced (was 0.5 loss → now 0.15)
        hp_delta = hero.current_health - prev_hp
        if hp_delta > 0:
            reward += hp_delta * 0.3   # gain: 0.3/HP (unchanged)
        else:
            capped = max(hp_delta, -20)
            reward += capped * 0.15    # loss: 0.15/HP (was 0.5), max −3 per step

        # Gold gained
        gold_delta = hero.current_gold - prev_gold
        if gold_delta > 0:
            reward += gold_delta * 0.02

        # Floor progress — raised from +10 to +25
        floor_delta = run.current_floor - prev_floor
        if floor_delta > 0:
            reward += floor_delta * 25.0

        # Enemy kill reward — direct combat engagement signal
        curr_alive = self._count_alive_enemies()
        enemies_killed = prev_alive - curr_alive
        if enemies_killed > 0:
            reward += enemies_killed * 5.0

        # Combat survival bonus
        now_in_combat = (run.current_state == GameState.InCombat)
        if prev_in_combat and not now_in_combat and hero.current_health > 0:
            reward += 2.0

        # Deck quality reward — fires once when a new card is picked
        current_deck_size = len(hero.deck.master_deck)
        if current_deck_size > self._prev_deck_size:
            new_card = hero.deck.master_deck[-1]
            star = getattr(new_card, 'star_rating', 1)
            if star >= 4:
                reward += 8.0
            elif star == 3:
                reward += 3.0
            # ★1–2: no penalty (early-game forced picks)
            self._prev_deck_size = current_deck_size

        # Light step cost
        reward -= 0.005

        # Terminal rewards
        if terminated:
            if hero.current_health > 0:
                reward += 500.0
            else:
                reward -= 200.0

        self._prev_enemy_count = curr_alive
        return obs, reward, terminated, truncated, info


json_path = os.path.join(DATASET_PATH, "python_roguelike", "GameData.json")
env = DefensiveEnv(seed=42, json_path=json_path)
obs, info = env.reset()
print("   DefensiveEnv v4 created — obs shape:", obs.shape)
print("   Action space:", env.action_space)
print("   Initial info:", info)

2026-03-21 09:40:15.499910: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774086015.771143      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774086015.862930      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774086016.534868      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774086016.534905      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774086016.534908      55 computation_placer.cc:177] computation placer alr

   DefensiveEnv created — obs shape: (153,)
   Action space: Discrete(67)
   Initial info: {'game_state': 'OnMap', 'floor': -1, 'hp': 60, 'max_hp': 60, 'gold': 100, 'deck_size': 10, 'relic_count': 1, 'step': 0}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [4]:
# ── 4. Validate Action Masking ───────────────────────────────────────────────
masked_env = ActionMasker(env, lambda e: e.action_masks())
obs, info = masked_env.reset()

for step in range(200):
    mask = masked_env.action_masks()
    valid_actions = np.where(mask)[0]
    action = np.random.choice(valid_actions)
    obs, reward, terminated, truncated, info = masked_env.step(action)
    if terminated or truncated:
        obs, info = masked_env.reset()

print(f" Masking validation passed — no illegal actions in 200 steps")

 Masking validation passed — no illegal actions in 200 steps


In [5]:
# ── 5. Vectorized Environment ─────────────────────────────────────────────────
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor

N_ENVS = 4

def make_env(seed_offset: int):
    def _init():
        _env = DefensiveEnv(
            seed=2000 + seed_offset,
            json_path=json_path,
            max_steps=10_000,
        )
        return ActionMasker(_env, lambda e: e.action_masks())
    return _init

vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)])
vec_env = VecMonitor(vec_env)
print(f" VecEnv ready: {N_ENVS} parallel envs")

2026-03-21 09:40:43.459597: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 09:40:43.459633: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 09:40:43.460115: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 09:40:43.460155: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774086043.508710     112 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already b

 VecEnv ready: 4 parallel envs


In [ ]:
# ── 6. MaskablePPO Model ─────────────────────────────────────────────────────
from sb3_contrib import MaskablePPO
from stable_baselines3.common.utils import get_linear_fn

policy_kwargs = dict(
    net_arch=dict(
        pi=[512, 512],
        vf=[512, 512],
    )
)

model = MaskablePPO(
    "MlpPolicy",
    vec_env,
    verbose=1,
    n_steps=2048,
    batch_size=512,
    n_epochs=10,
    learning_rate=get_linear_fn(3e-4, 1e-5, 1.0),
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.10,   # raised from 0.05 to prevent premature entropy collapse
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=policy_kwargs,
    tensorboard_log="/kaggle/working/tensorboard/defensive",
)

print("Model created. Policy params:", sum(p.numel() for p in model.policy.parameters()))

Using cpu device
Model created. Policy params: 717892


In [7]:
# ── 7. Callbacks ──────────────────────────────────────────────────────────────
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, CallbackList

eval_env = ActionMasker(
    DefensiveEnv(seed=8888, json_path=json_path),
    lambda e: e.action_masks()
)

checkpoint_cb = CheckpointCallback(
    save_freq=50_000,
    save_path="/kaggle/working/checkpoints/defensive",
    name_prefix="defensive_ppo",
    verbose=1,
)

eval_cb = EvalCallback(
    eval_env,
    best_model_save_path="/kaggle/working/best_model/defensive",
    log_path="/kaggle/working/eval_logs/defensive",
    eval_freq=25_000,
    n_eval_episodes=15,
    deterministic=False,
    verbose=1,
)

callbacks = CallbackList([checkpoint_cb, eval_cb])
print("   Callbacks configured")

   Callbacks configured


In [ ]:
# ── 8. Training ──────────────────────────────────────────────────────────────
TOTAL_TIMESTEPS = 5_000_000

print(f"   Starting Defensive Agent (v4) training for {TOTAL_TIMESTEPS:,} timesteps...")
print(f"   HP loss: 0.15/HP (was 0.5) | Floor: +25 (was +10) | Kill: +5")
print(f"   Network: [512, 512], LR: linear_schedule(3e-4), ent_coef: 0.10")
print()

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callbacks,
)

model.save("/kaggle/working/defensive_ppo_final")
print("\n Training complete! Model saved to: /kaggle/working/defensive_ppo_final")

   Starting Defensive Agent training for 10,000,000 timesteps...
   Strategy: Maximize HP retention, avoid damage, survive to end
   Win = floor 15 cleared (full game clear)
   Deck quality bonus: +2×(star-1) for ★3–5, −0.5 for ★1
   Network: [512, 512], LR: linear_schedule(3e-4), ent_coef: 0.05

Logging to /kaggle/working/tensorboard/defensive/MaskablePPO_1


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_monitor.VecMonitor object at 0x789dae938d10> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x789da4c32de0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 123      |
|    ep_rew_mean     | -30.9    |
| time/              |          |
|    fps             | 1420     |
|    iterations      | 1        |
|    time_elapsed    | 5        |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 133         |
|    ep_rew_mean          | -27.6       |
| time/                   |             |
|    fps                  | 989         |
|    iterations           | 2           |
|    time_elapsed         | 16          |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.009319568 |
|    clip_fraction        | 0.0824      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.49       |
|    explained_variance   | -0.000104   |
|    learning_rate        | 0.

KeyboardInterrupt: 

In [ ]:
# ── 9. Evaluation ────────────────────────────────────────────────────────────
model = MaskablePPO.load("/kaggle/working/best_model/defensive/best_model")

N_EVAL = 50
floors, full_wins, act_wins, hp_remainders = [], 0, 0, []

for ep in range(N_EVAL):
    ep_env = ActionMasker(DefensiveEnv(seed=ep, json_path=json_path), lambda e: e.action_masks())
    obs, info = ep_env.reset()
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True, action_masks=ep_env.action_masks())
        obs, reward, terminated, truncated, info = ep_env.step(int(action))
        done = terminated or truncated

    floors.append(info["floor"])
    if info["hp"] > 0 and terminated:
        hp_remainders.append(info["hp"] / info["max_hp"])
        if info["floor"] >= 15:
            full_wins += 1  # Full game clear (floor 15+)
        else:
            act_wins += 1   # Survived but didn't clear the game (act boss kill)

print(f"\n Defensive Agent Evaluation Results ({N_EVAL} episodes)")
print(f"═" * 50)
print(f"  Full-Game Win Rate:    {full_wins/N_EVAL*100:.1f}%  ({full_wins}/{N_EVAL}) — floor 15 cleared")
print(f"  Act Win Rate:          {act_wins/N_EVAL*100:.1f}%  ({act_wins}/{N_EVAL}) — survived but didn't finish")
print(f"  Death Rate:            {(N_EVAL-full_wins-act_wins)/N_EVAL*100:.1f}%")
print(f"  Avg Floor Reached:     {np.mean(floors):.1f} / 15")
print(f"  Max Floor Reached:     {max(floors)}")
if hp_remainders:
    print(f"  Avg HP% on Survive:    {np.mean(hp_remainders)*100:.1f}%")
    print(f"  Max HP% on Survive:    {max(hp_remainders)*100:.1f}%")